# 🧬 Práctica: Introducción a los Modelos Metabólicos

## Introducción

En biología de sistemas, uno de los retos más comunes es comprender y predecir el comportamiento de un microorganismo, ya sea en un entorno natural o en un biorreactor industrial. Aunque los experimentos de laboratorio proporcionan datos valiosos, la complejidad de las interacciones metabólicas a menudo requiere herramientas computacionales. Aquí, los modelos metabólicos a escala genómica (GEMs) — en particular, los que usan el marco de **COBRApy** — son fundamentales para analizar flujos de metabolitos, simular knockouts genéticos y diseñar estrategias de bioingeniería.

## Objetivos

- Cargar y explorar un modelo metabólico a
escala genómica (GEM) usando COBRApy.
- Inspeccionar la estructura del modelo: metabolitos, reacciones, genes y la matriz estequiométrica.
- Ejecutar un análisis de Flux Balance Analysis (FBA) e interpretar los resultados.
- Simular knockouts de genes y analizar su impacto en el crecimiento y los flujos.
- Diseñar una estrategia básica para redirigir flujos hacia un producto de interés.

## Requisitos previos

## 🤔 Preguntas guía para esta práctica

Tenga en mente estas preguntas mientras trabaja:

- **¿Cómo se traduce un genoma en un modelo matemático?**
- **¿Qué es exactamente "estado estacionario" y por qué importa?**
- **¿Cómo resuelve FBA qué flujos debe tener cada reacción?**
- **¿Por qué existe un trade-off entre crecimiento y producción?**
- **¿Cómo puedo usar matemática para diseñar cepas mejoradas?**

## Casos de estudio

### Caso A

| Característica   | Valor                              |
|:-----------------|:-----------------------------------|
| Organismo        | *Escherichia coli* core model      |
| Genes            | 137                                |
| Reacciones       | 95                                 |
| Metabolitos      | 72                                 |
| Referencia       | Orth et al., 2010, *Ecosal Plus*   |

### Caso B

| Característica   | Valor                              |
|:-----------------|:-----------------------------------|
| Organismo        | *Escherichia coli* K-12 MG1655     |
| Genes            | 1.366                              |
| Reacciones       | 2.583                              |
| Metabolitos      | 1.805                              |
| Referencia       | Orth et al., 2011, *Mol Syst Biol* |

> **Nota:** Todos los modelos son de libre acceso en [BiGG Models](http://bigg.ucsd.edu/).

## ⚙️ Paso 0 — Configurar el entorno

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 2–5 minutos.

In [ ]:
# Instalamos Conda para Colab

!pip install condacolab --quiet

import condacolab
condacolab.install() # Esto reinicia el kernel automáticamente

Tras el reinicio del kernel, ejecuta esta celda para confirmar que el entorno quedó correctamente configurado y para dejar registradas las versiones de las herramientas base (útil para citar en la sección de métodos de informes/artículos).

In [ ]:
# Verificar instalación de condacolab y registrar versiones del entorno base
import os
import sys
import subprocess

os.environ['PATH'] = '/usr/local/bin:/opt/conda/bin:' + os.environ['PATH']

# Instalar biopython
!pip install biopython --quiet

def version_pip(paquete):
    res = subprocess.run(["pip", "show", paquete], capture_output=True, text=True)
    for linea in res.stdout.splitlines():
        if linea.startswith("Version:"):
            return linea.split(":", 1)[1].strip()
    return "⚠️ no encontrado"

conda_version = subprocess.run(["conda", "--version"], capture_output=True, text=True).stdout.strip()

# google-colab no se instala vía pip estándar en el entorno conda (es parte
# de la integración interna del runtime), por lo que 'pip show' no lo detecta
# de forma confiable. Verificamos disponibilidad importando el módulo.
try:
    import google.colab  # noqa: F401
    colab_estado = "disponible (ejecutando en Google Colab)"
except ImportError:
    colab_estado = "no detectado (¿no estás en Google Colab?)"

print("📦 Versiones del entorno base:")
print(f"  - Python: {sys.version.split()[0]}")
print(f"  - {conda_version}")
print(f"  - biopython: {version_pip('biopython')}")
print(f"  - Google Colab: {colab_estado}")

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para modelas modelos a escala genomica y otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resultados mas facil para los humanos

In [ ]:
# Instalar y verificar el resultado
result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "matplotlib",
     "pandas",
     "numpy",
     "cobra",
     "cma",
     "--pre",
     "raven-toolbox",
     "geckopy",
     "--quiet"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"❌ Falló pip install:\n{result.stderr}")

# Imports centralizados
import json, glob, pathlib, gzip, shutil, re
from IPython.display import Image, display, SVG

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import raven_toolbox as rv
from raven_toolbox.manipulation import add_reactions_from_equations
from raven_toolbox.io import read_yaml_model
import geckopy
import cobra
from cobra import Model, Metabolite, Reaction
from cobra.io import load_model, read_sbml_model
from cobra.flux_analysis import production_envelope
from cobra.util.array import create_stoichiometric_matrix
from scipy.integrate import solve_ivp
from scipy.optimize import linprog
from tqdm import tqdm

os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

# Confirmar versiones realmente cargadas
print(f"✅ matplotlib {matplotlib.__version__}")
print(f"✅ pandas {pd.__version__}")
print(f"✅ numpy {np.__version__}")
print(f"✅ raven {rv.__version__}")
print(f"✅ cobra {cobra.__version__}")
# print(f"✅ gecko {geckopy.__version__}")
print(f"✅ geckopy v4.0.0b1") # While but reported is solved
print("✅ Dependencias Python listas")

Ahora vamos a definir algunas funciones que vamos a necesitar para mostrar el avance en desscargas y otros, e importar algunas librerias que usaremos leugo

In [ ]:
def ver_archivo(nombre, n_lineas=40, desde=0):
    print(f"\n{'='*15} {nombre} (líneas {desde+1}-{desde+n_lineas}) {'='*15}")
    with open(nombre, encoding="utf-8") as f:
        for i, linea in enumerate(f):
            if i < desde:
                continue
            if i >= desde + n_lineas:
                break
            print(linea.rstrip()[:160])

def fba_etanol(uptake_max):
    """Maximiza la excreción de etanol (R4) sujeto a S·v = 0 y 0 ≤ v1 ≤ uptake_max."""
    return linprog(c=[0, 0, 0, -1], A_eq=S_mini, b_eq=np.zeros(3),
                   bounds=[(0, uptake_max)] + [(0, None)]*3, method="highs")

def dfba(G0=10.0, t_fin=8.0, dt=0.05):
    """En cada paso: la cinética limita el uptake y el FBA reparte el flujo."""
    G, E, tt = G0, 0.0, 0.0
    hist = []
    while tt <= t_fin + 1e-9:
        ub = Vmax[0]*G/(Km[0]+G)                 # cota de uptake dada por Michaelis-Menten
        v_fba = fba_etanol(ub).x
        hist.append((tt, G, E, v_fba[0], v_fba[3]))
        G = max(G - v_fba[0]*dt, 0.0)
        E += v_fba[3]*dt
        tt += dt
    return np.array(hist)

def add_dynamic_bounds(model, y):
    """Use external concentrations to bound the uptake flux of glucose."""
    biomass, glucose = y[0], y[1]  # Extract biomass and glucose from the state vector
    glucose_max_import = -10 * glucose / (5 + glucose)
    model.reactions.EX_glc__D_e.lower_bound = glucose_max_import


def dynamic_system(t, y):
    """Calculate the time derivative of external species."""

    # Expand the boundary species and other external concentrations
    biomass, glucose, ethanol_ext, oxygen_ext, succinate_ext, lactate_ext, acetate_ext = y

    # Calculate the specific exchanges fluxes at the given external concentrations.
    with model:
        add_dynamic_bounds(model, y)

        cobra.util.add_lp_feasibility(model)
        feasibility = cobra.util.fix_objective_as_constraint(model)
        lex_constraints = cobra.util.add_lexicographic_constraints(
            model, ['Biomass_Ecoli_core',
                    'EX_glc__D_e',
                    'EX_etoh_e',
                    'EX_o2_e',
                    'EX_succ_e',
                    'EX_lac__D_e',
                    'EX_ac_e'],
                     ['max'] * 7) # Maximize all 7 objectives in lexicographic order

    # Get the specific rates for all species from the lexicographic solution.
    specific_growth_rate = lex_constraints.values[0]  # Biomass_Ecoli_core flux (1/h)
    specific_glucose_uptake_rate = lex_constraints.values[1]  # EX_glc__D_e flux (mmol/gDW/h)
    specific_ethanol_excretion_rate = lex_constraints.values[2] # EX_etoh_e flux (mmol/gDW/h)
    specific_oxygen_uptake_rate = lex_constraints.values[3] # EX_o2_e flux (mmol/gDW/h)
    specific_succinate_excretion_rate = lex_constraints.values[4] # EX_succ_e flux (mmol/gDW/h)
    specific_lactate_excretion_rate = lex_constraints.values[5] # EX_lac__D_e flux (mmol/gDW/h)
    specific_acetate_excretion_rate = lex_constraints.values[6] # EX_ac_e flux (mmol/gDW/h)

    # Calculate bulk exchange rates by multiplying by biomass concentration.
    # Note: COBRApy exchange fluxes are positive for uptake into the system, and negative for secretion.
    # So, for external concentrations: dC/dt = flux * biomass.
    db_dt = specific_growth_rate * biomass
    dg_dt = specific_glucose_uptake_rate * biomass
    detoh_dt = specific_ethanol_excretion_rate * biomass
    do2_dt = specific_oxygen_uptake_rate * biomass
    ds_dt = specific_succinate_excretion_rate * biomass
    dl_dt = specific_lactate_excretion_rate * biomass
    da_dt = specific_acetate_excretion_rate * biomass

    # This implementation is **not** efficient, so I display the current
    # simulation time using a progress bar.
    if dynamic_system.pbar is not None:
        dynamic_system.pbar.update(1)
        dynamic_system.pbar.set_description('t = {:.3f}'.format(t))

    return [db_dt, dg_dt, detoh_dt, do2_dt, ds_dt, dl_dt, da_dt] # Return all seven derivatives

dynamic_system.pbar = None


def infeasible_event(t, y):
    """
    Determine solution feasibility.

    Avoiding infeasible solutions is handled by solve_ivp's built-in event detection.
    This function re-solves the LP to determine whether or not the solution is feasible
    (and if not, how far it is from feasibility). When the sign of this function changes
    from -epsilon to positive, we know the solution is no longer feasible.

    """

    with model:

        add_dynamic_bounds(model, y)

        cobra.util.add_lp_feasibility(model)
        feasibility = cobra.util.fix_objective_as_constraint(model)

    return feasibility - infeasible_event.epsilon

infeasible_event.epsilon = 1E-6
infeasible_event.direction = 1
infeasible_event.terminal = True

## 📥 Parte 1 - Construir un modelo basico

Vamos a crear un modelo para que entendamos el funcionamiento de los modelos a escala genomica tal y como lo vimos en la sección teorica de este modulo.

Antes de avanzar en este punto primero vamos a entender los tipos de archivos que existen en la modelación a escala genomica.

Los modelos metabólicos a escala genómica (GEMs, por sus siglas en inglés) son representaciones matemáticas del metabolismo de un organismo. Incluyen las reacciones, los metabolitos, los genes y las relaciones gen-proteína-reacción (GPR) que conectan el genoma con el fenotipo. Para poder compartirlos, reproducirlos y analizarlos con distintas herramientas, estos modelos se almacenan en formatos de archivo estandarizados.

La elección del formato depende del objetivo: **SBML** es el estándar para el intercambio y la publicación; **JSON** es ligero y muy práctico en Python; **MAT** es el formato nativo de MATLAB; y **YAML** facilita el control de versiones y el trabajo colaborativo. Además, existen formatos complementarios, como FASTA o GenBank, que se utilizan como entrada para reconstruir modelos a partir de la secuencia del genoma, mismos que vimos en el modulo 1 de este curso.

En este cuaderno se trabaja principalmente con `cobrapy`, `RAVEN` y `GECKO`. La siguiente tabla resume los formatos más comunes, sus extensiones y las herramientas con las que se usan.

| Formato | Extensión | Descripción | Herramientas |
|---|---|---|---|
| **SBML** | `.xml`, `.sbml` | Estándar de intercambio (con extensión FBC) | COBRApy, COBRA Toolbox, RAVEN |
| **JSON** | `.json` | Ligero, nativo de BiGG | COBRApy, BiGG, Escher |
| **MAT** | `.mat` | Estructura nativa de MATLAB | COBRA Toolbox, RAVEN |
| **YAML** | `.yml`, `.yaml` | Legible, ideal para Git | RAVEN, GECKO, Yeast-GEM |
| **Excel** | `.xlsx`, `.xls` | Curación manual de reacciones y genes | RAVEN, COBRA Toolbox |
| **Tabular** | `.tsv`, `.csv`, `.txt` | Listas de reacciones o metabolitos | pandas, scripts propios |
| **Pickle** | `.pkl` | Objeto Python serializado, no portable | COBRApy |
| **FASTA** | `.fasta`, `.faa` | Secuencias de proteínas del genoma | RAVEN, CarveMe, gapseq |
| **GenBank / GFF** | `.gbk`, `.gff` | Anotación del genoma | ModelSEED, CarveMe |
| **BioPAX** | `.owl` | Rutas y redes biológicas | Pathway Tools, Cytoscape |
| **MEMOTE** | `.json`, `.html` | Informe de calidad del modelo | MEMOTE |

### Recomendaciones
- **Intercambio y publicación:** SBML nivel 3 con FBC v2.
- **Control de versiones y trabajo en equipo:** YAML.
- **Uso en Python:** JSON o SBML.
- **Uso en MATLAB:** `.mat`, o SBML/YAML mediante RAVEN.

In [ ]:
YEAST_GEM = "v9.1.0"   # la versión que documenta RAVEN
BASE = f"https://raw.githubusercontent.com/SysBioChalmers/yeast-GEM/{YEAST_GEM}/model/"
ARCHIVOS = ["yeast-GEM.yml", "yeast-GEM.xml"]

for nombre in ARCHIVOS:
    if not os.path.exists(nombre):
        subprocess.run(["curl", "-sSL", "--fail", "-o", nombre, BASE + nombre], check=True)
    with open(nombre, "rb") as f:
        inicio = f.read(60).decode("utf-8", errors="replace")
    print(f"✅ {nombre}: {os.path.getsize(nombre)/1e6:.2f} MB | inicio: {inicio!r}")

Ahora que hemos cargado los modelos, vamos a comparar si tienen la misma información

In [ ]:
m_yaml = read_yaml_model("yeast-GEM.yml") # YAML de RAVEN → raven-toolbox
m_sbml = read_sbml_model("yeast-GEM.xml") # SBML → cobrapy

for etiqueta, m in [("YAML (raven-toolbox)", m_yaml), ("SBML (cobrapy)", m_sbml)]:
    print(f"📊 {etiqueta}")
    print(f"   Reacciones : {len(m.reactions)}")
    print(f"   Metabolitos: {len(m.metabolites)}")
    print(f"   Genes      : {len(m.genes)}")
    print(f"   Tipo objeto: {type(m).__module__}.{type(m).__name__}\n")

¿Contienen las mismas reacciones?

In [ ]:
ids_yaml = {r.id for r in m_yaml.reactions}
ids_sbml = {r.id.removeprefix("R_") for r in m_sbml.reactions}   # SBML añade el prefijo R_
print("Reacciones en común      :", len(ids_yaml & ids_sbml))
print("Solo en YAML             :", len(ids_yaml - ids_sbml))
print("Solo en SBML             :", len(ids_sbml - ids_yaml))

# Una misma reacción en ambos
rid = sorted(ids_yaml & ids_sbml)[10]
r_y = m_yaml.reactions.get_by_id(rid)
r_s = m_sbml.reactions.get_by_id("R_" + rid) if ("R_" + rid) in m_sbml.reactions else m_sbml.reactions.get_by_id(rid)

print(f"\n--- {rid} ---")
print("YAML :", r_y.reaction, "|", r_y.bounds, "|", r_y.gene_reaction_rule)
print("SBML :", r_s.reaction, "|", r_s.bounds, "|", r_s.gene_reaction_rule)

# FBA rápido con cada formato
print("\nCrecimiento (FBA):")
print("  YAML:", m_yaml.slim_optimize())
print("  SBML:", m_sbml.slim_optimize())

Ahora vamos a crear un modelo desde cero (`toy_model`). El modelo es similar al que trabajamos en la parte teorica de este curso.

[Ver figura 2](https://doi.org/10.3389/fmicb.2016.00907)

In [ ]:
modelo = Model("toy_model")

# 1) Metabolitos (todos en el citosol, como en la figura)
modelo.add_metabolites([
    Metabolite(m, name=f"Metabolito {m}", compartment="c") for m in "ABCDE"
])
print(len(modelo.metabolites), "metabolitos")

# 2) Reacciones internas a partir de ecuaciones (RAVEN calcula la estequiometría)
#    "=>" irreversible, "<=>" reversible; "bounds" sobreescribe la flecha.
add_reactions_from_equations(
    modelo,
    [
        {"id": "r1", "name": "Reaction 1", "equation": "A + D => B",   "gene_reaction_rule": "g1"},
        {"id": "r2", "name": "Reaction 3", "equation": "2 B <=> 2 E",  "gene_reaction_rule": "g2"},
        {"id": "r3", "name": "Reaction 4", "equation": "E => A",       "gene_reaction_rule": "g3"},
        {"id": "r4", "name": "Reaction 5", "equation": "B + E => 2 C", "gene_reaction_rule": "g4"},
    ],
    allow_new_mets=False,    # un error de escritura en un metabolito dará error, no uno nuevo
    allow_new_genes=True,    # g1..g4 se crean al nombrarlos
)

# 3) Transportes r5, r6, r7: reacciones normales con un solo metabolito
#    (positivo = entrada a la célula, negativo = salida)
transportes = {
    "r5": ("A", "transporte de A (salida)",  (-1000, 0)),
    "r6": ("D", "transporte de D (cerrado)", (0, 0)),
    "r7": ("E", "transporte de E (entrada)", (0, 10)),
}
for rid, (met, nombre, (lb, ub)) in transportes.items():
    r = Reaction(rid, name=nombre, lower_bound=lb, upper_bound=ub)
    modelo.add_reactions([r])
    r.add_metabolites({modelo.metabolites.get_by_id(met): 1})

# 4) Biomasa: demanda de C (C →). 'demand' no exige compartimento externo
modelo.add_boundary(modelo.metabolites.C, type="demand", reaction_id="biomass")

# 5) Leer las ecuaciones de vuelta (equivalente a constructEquations)
print(f"\n✅ {len(modelo.reactions)} reacciones, {len(modelo.metabolites)} metabolitos, {len(modelo.genes)} genes")
for r in modelo.reactions:
    print(f"  {r.id:8s} {r.reaction:20s} bounds={r.bounds}  GPR={r.gene_reaction_rule or '-'}")

El modelo puede seguir ajustandose según sea el caso. Por ejemplo, vamos a agregar una nueva reacción.

In [ ]:
# Metabolito nuevo
modelo.add_metabolites([Metabolite("F", name="Metabolito F", compartment="c")])

# Reacción nueva desde una ecuación
add_reactions_from_equations(
    modelo,
    [{"id": "r9", "name": "A + E → F", "equation": "A + E => F"}],
    allow_new_mets=False,
)

# Salida de F
modelo.add_boundary(modelo.metabolites.F, type="demand", reaction_id="r10")

print(modelo.reactions.r9.reaction, "|", modelo.reactions.r10.reaction)

S2 = create_stoichiometric_matrix(modelo, array_type="DataFrame")
display(S2.astype(int))

In [ ]:
# Comprobación: en estado estacionario S·v = 0
# Matriz estequiométrica del modelo (DataFrame), construida directamente desde el modelo
S_modelo = pd.DataFrame(
    0.0,
    index=[m.id for m in modelo.metabolites],
    columns=[r.id for r in modelo.reactions],
)
for r in modelo.reactions:
    for met, coef in r.metabolites.items():
        S_modelo.loc[met.id, r.id] = coef

print("S·v = 0 →", np.allclose(S_modelo.values @ v.values, 0, atol=1e-6))

Bien, ahora resolvamos el modelo y encontremos el vector de soluciones (flujos vj). En la parte 3 de este ejercicio abordaremos con mayor detalle que es un FBA y como se resuelve. Por ahora miremos generalidades

In [ ]:
modelo.objective ="biomass"

solucion = modelo.optimize()

print(f"Estado             : {solucion.status}")
print(f"Objetivo (Biomasa) : {solucion.objective_value:.2f}")
print(f"Entrada            : {solucion.fluxes['r7']:.2f}")
print(f"Salida             : {solucion.fluxes['r5']:.2f}")

In [ ]:
# Tabla de flujos
display(solucion.fluxes.to_frame("flujo").round(3))

## 📘 Parte 2 - Cargar y explorar el modelo

### 2.1 Cargar el modelo

El modelo *core* de *E. coli* representa las vías metabólicas centrales.
Es ideal para aprender porque es pequeño y bien documentado.
Las mismas operaciones aplican para modelos completos como iJO1366.

**Preguntas orientadoras:**
- ¿Cuántas reacciones diferentes ocurren en una célula?
- ¿Cómo se puede representar esto matemáticamente?
- ¿Por qué el modelo "core" es más pequeño que iJO1366?


In [ ]:
model_to_use = "textbook"

model = load_model(model_to_use) # puede reemplazar por iJO1366

print("=" * 50)
print(f"Modelo:                {model.id}")
print(f"Número de reacciones:  {len(model.reactions)}")
print(f"Número de metabolitos: {len(model.metabolites)}")
print(f"Número de genes:       {len(model.genes)}")
print("=" * 50)

model.summary()

### 2.2 Explorar los metabolitos

**Preguntas antes de ejecutar:**
- ¿Qué metabolitos crees que serán los más abundantes?
- ¿Cuál es la diferencia entre un metabolito en el citoplasma vs. el exterior?
- ¿Por qué algunos metabolitos tienen múltiples reacciones mientras otros solo una?

In [ ]:
print(f"{'ID':<20} {'Nombre':<35} {'Fórmula':<15} {'Compartimento'}")
print("-" * 85)
for met in list(model.metabolites)[:10]:
    print(f"{met.id:<20} {met.name:<35} {met.formula:<15} {met.compartment}")

In [ ]:
# Sufijos de compartimento: _c = citoplasma, _e = extracelular, _p = periplasma
glucosa = model.metabolites.get_by_id("glc__D_e")

print(f"Metabolito:    {glucosa.id}")
print(f"Nombre:        {glucosa.name}")
print(f"Fórmula:       {glucosa.formula}")
print(f"Carga:         {glucosa.charge}")
print(f"Compartimento: {glucosa.compartment}")
print()
print(f"Participa en {len(glucosa.reactions)} reacción(es):")
for rxn in glucosa.reactions:
    print(f"  -> {rxn.id}: {rxn.name}")

In [ ]:
# Ahora como un metabolito se incorpora en multiples reacciones
model.metabolites.nadh_c.summary()

Análisis: Glucosa y sus reacciones

- ¿Por qué la glucosa aparece en varias reacciones?
- ¿Cuáles son los puntos de entrada/salida de glucosa?
- ¿Todas las reacciones la consumen o también la producen?


### 2.3 Explorar las reacciones

**Preguntas orientadoras:**
- ¿Qué significan los límites `lb` y `ub`?
- ¿Por qué algunas reacciones tienen límites asimétricos?
- ¿Qué diferencia hay entre una reacción reversible e irreversible?

In [ ]:
print(f"{'ID':<15} {'Nombre':<30} {'lb':>8} {'ub':>8}")
print("-" * 65)
for rxn in list(model.reactions)[:10]:
    print(f"{rxn.id:<15} {rxn.name:<30} {rxn.lower_bound:>8.1f} {rxn.upper_bound:>8.1f}")

In [ ]:
# PFK: Fosfofructoquinasa — reacción clave de la glucólisis
pfk = model.reactions.get_by_id("PFK")

print(f"Reacción:   {pfk.id}")
print(f"Nombre:     {pfk.name}")
print(f"Ecuación:   {pfk.reaction}")
print(f"Límites:    [{pfk.lower_bound}, {pfk.upper_bound}]  mmol/gDW/h")
print(f"Reversible: {pfk.reversibility}")
print(f"\nRegla GPR: {pfk.gene_reaction_rule}")
print("\nEstequiometría:")
for met, coeff in pfk.metabolites.items():
    d = "CONSUME" if coeff < 0 else "PRODUCE"
    print(f"  {d}: {abs(coeff)} x {met.id} ({met.name})")

Análisis: Entendiendo la regla GPR

- ¿Qué significa esta regla?
- ¿Qué pasaría si uno de estos genes se deletara?
- ¿La reacción sería completamente bloqueada o parcialmente?

### 2.4 La matriz estequiométrica S

In [ ]:
S = create_stoichiometric_matrix(model)

print(f"Dimensiones de S: {S.shape}")
print(f"  Filas    = {S.shape[0]} metabolitos")
print(f"  Columnas = {S.shape[1]} reacciones")
print(f"  Densidad = {np.count_nonzero(S) / S.size * 100:.1f}% de elementos no cero")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.spy(S, markersize=0.5, color='steelblue')
ax.set_xlabel('Reacciones', fontsize=12)
ax.set_ylabel('Metabolitos', fontsize=12)
ax.set_title('Matriz Estequiométrica S\n(cada punto = coeficiente distinto de cero)', fontsize=13)
plt.tight_layout()
plt.show()

## 🌉  Parte 3 - De las ODEs al FBA: una misma ecuación

Antes de usar FBA (que lo hace automáticamente), vamos a entender qué está sucediendo detrás de escenas.

Un modelo metabólico, cinético o no, parte del mismo balance de masa:

$$\frac{d\mathbf{x}}{dt} = \mathbf{S}\cdot\mathbf{v}$$

- **ODEs (cinético):** cada flujo depende de las concentraciones, $v = f(\mathbf{x})$ (Michaelis-Menten). Hay que conocer $V_{max}$ y $K_m$ de cada enzima.
- **FBA:** se asume estado estacionario, $d\mathbf{x}/dt = 0$, es decir $\mathbf{S}\cdot\mathbf{v}=0$. No hace falta conocer las cinéticas: se reemplazan por cotas ($lb \le v \le ub$) y por un objetivo (maximizar un flujo).

El sistema (mini modelo) G → 2P → E de la sección 5 es la parte *interna* (R2 y R3) del mini-modelo de la Parte 2. Las reacciones de frontera R1 (uptake de glucosa) y R4 (excreción de etanol) completan la matriz $\mathbf{S}$ de 3×4.

> Supuesto: se usa 1 gDW/L de biomasa para poder combinar flujos en mmol/gDW/h con concentraciones en mmol/L.

**Preguntas de anticipación:**
- ¿Recuerdas que significa que S·v = 0?
- ¿Por qué el estado estacionario es importante en modelación?
- ¿Cómo se podría optimizar (maximizar) la producción de etanol bajo esta restricción?
- ¿Cuántas soluciones posibles crees que hay para S·v = 0?


### 3.1. Reacciones del modelo como matriz matematica

Reacciones del mini-modelo:
- R1: Glucosa externo → Glucosa citoplasma    (uptake)
- R2: Glucosa citoplasma → 2 Piruvato         (glucólisis simplificada)
- R3: Piruvato → Etanol + CO2                 (fermentación)

Metabolitos (solo INTERNOS para estado estacionario):
- G (Glucose)
- P (Pyruvate)
- E (Etanol)  

Matriz estequiométrica S (metabolitos internos x reacciones):

|    | R1 | R2 | R3 | R4 |
|:---|:---|:---|:---|:---|
| G  | +1 | -1 | 0  | 0  |
| P  | 0  | +2 | -1 | 0  |
| E  | 0  | 0  | +1 | -1 |


In [ ]:
# Misma matriz de la práctica: filas = Glc_cit, Pyr, Etoh_cit; columnas = R1..R4
S_mini = np.array([[ 1, -1,  0,  0],
                   [ 0,  2, -1,  0],
                   [ 0,  0,  1, -1]])

# La parte interna (R2 y R3) es exactamente el sistema G → 2P → E de la teoría
S_ode = S_mini[:, 1:3]
print("S de la ODE (filas G, P, E; columnas v1, v2):\n", S_ode)

### 3.2. Maximizar etanol (objetivo biológico)

Vamos a resolver el sistemas de ODEs. Pero antes:

- ¿Cuál crees que será el máximo etanol producido?"
- ¿Qué limitará la producción: la glucosa disponible u otra cosa?"
- ¿Cuál es la relación óptima entre los flujos v_1, v_2?"


In [ ]:
# Cinética de Michaelis-Menten (parámetros de la teoría)
Vmax = np.array([2.5, 5.0])
Km   = np.array([0.5, 0.2])

def tasas(x):
    G, P, E = np.maximum(x, 0)
    return np.array([Vmax[0]*G/(Km[0]+G), Vmax[1]*P/(Km[1]+P)])

def rhs(t, x):
    return S_ode @ tasas(x)          # dx/dt = S·v(x)

sol = solve_ivp(rhs, (0, 10), [10.0, 0.0, 0.0], dense_output=True, rtol=1e-8, atol=1e-10)

# Tabla (reemplaza la tabla manual de la teoría)
t_tab = [0, 0.5, 1, 1.5, 2, 3, 4, 5, 6, 7, 8]
filas = []
for t in t_tab:
    G, P, E = sol.sol(t)
    v1, v2 = tasas([G, P, E])
    filas.append({"t (h)": t, "[G]": G, "[P]": P, "[E]": E, "v1": v1, "v2": v2, "2G+P+E": 2*G + P + E})
display(pd.DataFrame(filas).round(3))

Grafiquemos el comportamiento del sistema

In [ ]:
# Gráficas
t = np.linspace(0, 8, 800)
G, P, E = sol.sol(t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t, G, label="Glucosa [G]")
ax1.plot(t, P, label="Piruvato [P]")
ax1.plot(t, E, label="Etanol [E]")
ax1.set_xlabel("Tiempo (h)"); ax1.set_ylabel("Concentración (mmol/L)")
ax1.set_title("Dinámica del sistema (ODE)"); ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(t, 2*G + P + E, color="k")
ax2.set_ylim(0, 25)
ax2.set_xlabel("Tiempo (h)"); ax2.set_ylabel("2[G] + [P] + [E] (mmol/L)")
ax2.set_title("Conservación de masa: la suma se mantiene en 20")
ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

La gráfica de la derecha muestra por qué el etanol final tiene que ser 20: el rendimiento 2:1 lo fija la estequiometría, no la cinética.

### 3.3 ¿Dónde se parece la ODE al estado estacionario (S·v = 0)?

El problema de programación lineal es:

    max     v_2  (flujo de etanol - R4)
    s.a.    S·v = 0
            v_1 ≥ 0  (uptake no puede ser negativo)
            v_1 ≤ 10 (límite de uptake)
            v_i ≥ 0  para todo i

In [ ]:
X = sol.sol(t)
v = np.array([tasas(X[:, i]) for i in range(len(t))])     # columnas: v1 (glucólisis), v2 (fermentación)
dP = 2*v[:, 0] - v[:, 1]                                   # dP/dt = 2·v1 − v2

# Cuasi-estado estacionario: dP/dt pequeño respecto al flujo que entra al pool de piruvato
casi = (np.abs(dP) < 0.05 * 2*v[:, 0]) & (v[:, 0] > 0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(t, X[1], color="tab:orange", label="[P]")
ax1.set_xlabel("Tiempo (h)"); ax1.set_ylabel("[P] (mmol/L)")
ax1.set_title("Piruvato: transitorio y cuasi-estado estacionario"); ax1.grid(alpha=.3)

ratio = np.where(v[:, 0] > 1e-3, v[:, 1] / np.maximum(v[:, 0], 1e-12), np.nan)
ax2.plot(t, ratio, color="tab:green", label="v2 / v1 (ODE)")
ax2.axhline(2, color="r", ls="--", label="Razón que impone S·v = 0")
ax2.set_ylim(0, 3.5)
ax2.set_xlabel("Tiempo (h)"); ax2.set_ylabel("v2 / v1")
ax2.set_title("El acoplamiento de flujos converge a la estequiometría")
ax2.legend(); ax2.grid(alpha=.3)

if casi.any():
    t0, t1 = t[casi].min(), t[casi].max()
    for ax in (ax1, ax2):
        ax.axvspan(t0, t1, color="gray", alpha=.2)
    ax1.text(t0, ax1.get_ylim()[1]*0.9, " cuasi-estacionario", fontsize=9)
    print(f"Ventana cuasi-estacionaria: t ≈ {t0:.1f}–{t1:.1f} h")

plt.tight_layout(); plt.show()

# Con v_uptake = v1 y v_excreción = v2, el residuo S·v solo queda en el piruvato
v_full = np.column_stack([v[:, 0], v[:, 0], v[:, 1], v[:, 1]])   # [R1, R2, R3, R4]
residuo = (S_mini @ v_full.T).T                                    # columnas: Glc_cit, Pyr, Etoh_cit
print("Residuo máximo |S·v| en la ventana cuasi-estacionaria:",
      np.abs(residuo[casi]).max().round(3) if casi.any() else "sin ventana")
print("El residuo de Pyr coincide con dP/dt:", np.allclose(residuo[:, 1], dP))

#### 📖 Cómo leer esta gráfica

Imagina un **lavamanos**: la llave es la glucólisis ($v_1$), el desagüe es la fermentación ($v_2$) y el nivel del agua es la concentración de piruvato [P]. Por cada glucosa que consume la llave se producen **2 piruvatos**, así que el agua entra a razón de $2\,v_1$ y sale a razón de $v_2$.

**Panel izquierdo: el nivel de piruvato [P]**

| Fase | Qué ocurre | Por qué |
|:--|:--|:--|
| **Inicio (0 → ~2 h)** | [P] sube | La llave ya echa agua, pero el desagüe saca poco porque el lavamanos está casi vacío. |
| **Zona gris (~2 → 3.5 h)** | [P] se mantiene casi constante | El desagüe saca justo lo que entra: $2\,v_1 \approx v_2$. Hay flujo, pero el nivel no cambia. **Este es el estado estacionario.** |
| **Final (> 4 h)** | [P] cae a 0 | Se acaba la glucosa, la llave se cierra y el desagüe vacía lo que queda. |

**Panel derecho: la razón $v_2 / v_1$**

Para que [P] no cambie se necesita $2\,v_1 = v_2$, es decir, $v_2 / v_1 = 2$. Esa es la **línea roja punteada**, el valor que exige $\mathbf{S}\cdot\mathbf{v}=0$. La línea verde es la razón que realmente tiene la ODE:

- **Al inicio (≈ 1.7 a las 0.5 h):** está por debajo de 2. El desagüe todavía no alcanza a la llave, por eso [P] sube.
- **En la zona gris (≈ 2):** la línea verde cruza la roja. Aquí se cumple $\mathbf{S}\cdot\mathbf{v}\approx 0$.
- **Al final (> 2):** sube por encima de 2 porque la llave se está cerrando mientras el desagüe aún vacía el piruvato acumulado.

**Qué imprimen las líneas de texto**

- *Ventana cuasi-estacionaria*: el intervalo de tiempo donde [P] está plano (la zona gris).
- *Residuo máximo |S·v|*: cuánto se acumula o falta de cada metabolito. En la zona gris es casi 0.
- *El residuo de Pyr coincide con dP/dt*: confirma que $\mathbf{S}\cdot\mathbf{v}$ es la pendiente de la curva naranja. Donde la curva es plana, el residuo es 0.

**🎯 Conclusión**

El **FBA** resuelve solo la zona gris: supone que la llave y el desagüe están equilibrados desde el primer instante, sin necesidad de conocer $V_{max}$ ni $K_m$. La **ODE** cuenta toda la historia: el llenado del inicio, el equilibrio y el vaciado final.

Por eso el FBA acierta en el *rendimiento* (siempre 2 etanoles por glucosa), pero no puede predecir *cuánto tarda* el sistema en llegar al equilibrio. Para eso hacen falta las cinéticas.

### 3.4. El mismo sistema resuelto como FBA (LP)

In [ ]:
res = fba_etanol(10)
print("Flujos óptimos [R1, R2, R3, R4]:", res.x.round(3))
print("Etanol máximo:", round(-res.fun, 3), "(2 etanoles por cada glucosa)")

# Predicción del FBA al variar la cota de uptake
cotas = np.linspace(0, 10, 11)
e_max = [-fba_etanol(u).fun for u in cotas]

plt.figure(figsize=(6, 4.2))
plt.plot(cotas, e_max, "o-")
plt.xlabel("Cota de uptake de glucosa (mmol/gDW/h)")
plt.ylabel("Etanol máximo (mmol/gDW/h)")
plt.title("Predicción del FBA: etanol = 2 × uptake")
plt.grid(alpha=.3); plt.show()

El FBA da v = [10, 10, 20, 20] y una pendiente de 2 que no depende de ningún $V_{max}$ ni $K_m$.

**Análisis:**

- ¿Por qué este es el máximo de ethanol?
- ¿Por qué v_1 = 10 (el máximo permitido)?
- ¿Qué pasaría si permitiéramos v_1 > 10?
- ¿Por qué v₁ = v₂ = 10, pero v₃ = v₄ = 20? (¿Acoplamiento de flujos?)
- ¿Podrías diseñar un experimento para validar esta predicción?

### 3.5. FBA dinámico (dFBA): el puente entre los dos mundos

Las ODEs saben **cuánto tarda** cada cosa, pero necesitan $V_{max}$ y $K_m$ de cada enzima. El FBA solo necesita la estequiometría, pero describe un **instante** en estado estacionario. El **FBA dinámico (dFBA)** combina ambos:

1. **La cinética solo pone el límite de entrada:** el uptake de glucosa sigue Michaelis-Menten, $ub = \dfrac{V_{max}\,[G]}{K_m + [G]}$.
2. **El FBA reparte el flujo:** con esa cota, se resuelve $\mathbf{S}\cdot\mathbf{v}=0$ y se maximiza el objetivo.
3. **Se actualiza el medio:** con los flujos obtenidos, se avanza un paso de tiempo (Euler): $[X]_{n+1} = [X]_n + v \cdot \Delta t$.
4. **Se repite** con las nuevas concentraciones.

Así el FBA no necesita conocer las cinéticas internas (solo la del uptake), y aun así se obtienen curvas en el tiempo. En la celda siguiente se compara con la ODE completa: el resultado final coincide (20 mmol/L de etanol), y la diferencia está en los primeros instantes, porque el FBA no ve el transitorio del piruvato.

In [ ]:
h = dfba()
t_f, G_f, E_f, up_f, et_f = h.T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(t, G, label="[G] ODE"); ax1.plot(t_f, G_f, "--", label="[G] FBA dinámico")
ax1.plot(t, E, label="[E] ODE"); ax1.plot(t_f, E_f, "--", label="[E] FBA dinámico")
ax1.set_xlabel("Tiempo (h)"); ax1.set_ylabel("Concentración (mmol/L)")
ax1.set_title("ODE vs FBA dinámico"); ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(t, v[:, 1], label="v2: producción de etanol (ODE)")
ax2.plot(t_f, et_f, "--", label="R4: etanol (FBA dinámico)")
ax2.set_xlabel("Tiempo (h)"); ax2.set_ylabel("Flujo (mmol/gDW/h)")
ax2.set_title("El FBA no ve el transitorio inicial"); ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()
print(f"Etanol final → ODE: {E[-1]:.2f} | FBA dinámico: {E_f[-1]:.2f} mmol/L")

📖 **Cómo leer esta gráfica**

Las líneas continuas son la **ODE** (cinética completa) y las punteadas son el **FBA dinámico**. Los dos usan la misma cinética para el uptake de glucosa; la diferencia es que el FBA no tiene un pool de piruvato que llenar.

**Panel izquierdo: concentraciones en el tiempo**

- **Glucosa [G]:** las dos curvas casi se superponen. La glucosa solo depende del uptake, y ese lo controla la misma ecuación de Michaelis-Menten en ambos métodos, así que no hay diferencia.
- **Etanol [E]:** la curva punteada (FBA) va **un poco por encima** de la continua (ODE) durante casi todo el recorrido, y ambas terminan en **20 mmol/L**. Por ejemplo, a las 3 h el FBA marca ≈ 13.8 y la ODE ≈ 12.4.

**¿Por qué el FBA va adelantado?** El FBA supone estado estacionario desde el primer instante: cada glucosa que entra se convierte de inmediato en 2 etanoles. En la ODE, el piruvato primero tiene que acumularse en un pool intermedio antes de convertirse en etanol, así que el etanol llega con retraso. La brecha entre las dos curvas de etanol es **exactamente el piruvato acumulado** en ese momento (≈ 1.4 mmol/L a las 3 h). Esto sale de la conservación de masa: $2[G] + [P] + [E] = 20$ en la ODE y $2[G] + [E] = 20$ en el FBA.

**Panel derecho: velocidad de producción de etanol**

- **ODE (continua):** empieza en **0**, porque aún no hay piruvato, sube hasta ≈ 4.4 y luego cae.
- **FBA (punteada):** empieza de golpe en ≈ 4.8 ($= 2 \times v_1$) y baja de forma suave a medida que se agota la glucosa.
- **Hacia el final:** la ODE queda **por encima** del FBA, porque está vaciando el piruvato que acumuló antes. Esa "deuda" se paga aquí.

Las dos curvas encierran la **misma área**: el etanol total producido es igual (20 mmol/L). Lo que cambia es *cuándo* se produce.

🎯 **Conclusión**

| | ODE | FBA dinámico |
|:--|:--|:--|
| Necesita $V_{max}$ y $K_m$ de cada enzima | Sí | Solo del uptake |
| Captura el transitorio (llenado de pools) | Sí | No |
| Rendimiento final (2 etanoles por glucosa) | 20 | 20 |
| Costo de cálculo | Bajo | Mayor (un LP por paso) |

El dFBA es una aproximación **cuasi-estacionaria**: acierta en el balance global y en la tendencia, y falla en los primeros instantes y en los intercambios rápidos de pools. Funciona bien cuando los metabolitos intermedios son pequeños y rápidos respecto al cambio del medio, que es la situación típica de los modelos a escala genómica.

### Análisis final

**Ideas centrales**
- ¿Por qué el FBA predice etanol alto desde t = 0 y la ODE no? (el pool de piruvato)
- ¿Qué información necesita la ODE que el FBA no necesita?
- ¿Qué se pierde al asumir estado estacionario?
- ¿Por qué las dos metodologías terminan en 20 mmol/L de etanol, aunque la dinámica sea distinta? (conservación de masa)
- ¿Qué significa que $\mathbf{S}\cdot\mathbf{v}=0$ y qué le pasa a un metabolito cuyo residuo no es cero?

**Validez del supuesto**
- ¿En qué momento de la simulación el estado estacionario es una buena aproximación y cuándo falla?
- ¿Qué características debe tener un metabolito para que se pueda tratar como estacionario? (pool pequeño y reacciones rápidas frente al cambio del medio)
- Si el piruvato fuera un pool muy grande (por ejemplo, $K_m$ de la fermentación mucho mayor), ¿el FBA seguiría siendo una buena aproximación?

**Experimentos numéricos (cambia los parámetros y vuelve a ejecutar)**
- Reduce $V_{max,2}$ por debajo de $2\,v_1$. ¿Qué le pasa al piruvato? ¿Sigue existiendo una zona cuasi-estacionaria? ¿Y qué predice el FBA?
- Duplica la glucosa inicial. ¿Cambia el rendimiento (etanol por glucosa)? ¿Y el tiempo total del proceso?
- Aumenta $V_{max,1}$. ¿Qué cambia en la ODE y qué cambia (o no) en la predicción del FBA?

**Del modelo pequeño al modelo completo**
- ¿Qué metabolitos y reacciones habría que agregar para que el mini-modelo pudiera predecir crecimiento?
- ¿Qué hace la función objetivo en el FBA y qué pasaría si se eligiera otra (por ejemplo, minimizar el uptake de glucosa)?
- En un modelo de miles de reacciones, ¿sería viable conocer todos los $V_{max}$ y $K_m$ que pide una ODE? ¿Por qué el FBA sí escala?
- ¿Qué tipo de resultados esperarías del dFBA con el modelo completo que no podrías obtener con el FBA estático?

**Pensamiento crítico**
- Un resultado del FBA, ¿es una predicción o una cota teórica? ¿Cómo lo compararías con datos experimentales?
- ¿Qué experimento mediría el piruvato intracelular para comprobar la zona cuasi-estacionaria?

## 📊 Parte 4. FBA básico — maximizar crecimiento en glucosa

**Pregunta:** ¿a qué velocidad máxima puede crecer *E. coli* consumiendo glucosa como única fuente de carbono?

**Preguntas de anticipación:**
- ¿Cuál crees que será la tasa de crecimiento máxima?
- ¿Qué reacciones crees que tendrán los flujos más altos?
- ¿Cómo se compara esto con el mini-modelo?
- ¿Por qué el modelo predice crecimiento cuando el mini-modelo no?

In [ ]:
model = load_model(model_to_use)

model.objective = {model.reactions.Biomass_Ecoli_core: 1}

solution = model.optimize()

print("=" * 50)
print("Resultado del FBA")
print("=" * 50)
print(f"Estado:              {solution.status}")
print(f"Tasa de crecimiento: {solution.objective_value:.4f} 1/h")
print(f"Flujo de biomasa:    {solution.fluxes['Biomass_Ecoli_core']:.4f} mmol/gDW/h")
print(f"{'Reacción':<15} {'Nombre':<35} {'Flujo (mmol/gDW/h)':>20}")
print("-" * 75)

fluxes_sorted = solution.fluxes.abs().sort_values(ascending=False)
for rxn_id in fluxes_sorted.index[:20]:
    rxn = model.reactions.get_by_id(rxn_id)
    flux_val = solution.fluxes[rxn_id]
    if abs(flux_val) > 1e-6:
        print(f"{rxn_id:<15} {rxn.name[:34]:<35} {flux_val:>20.4f}")

**Análisis:**

- ¿Por qué biomasa tiene el flujo más alto?
- ¿Qué reacciones de glucólisis ves en la lista?
- ¿Qué significa que una reacción tenga flujo negativo?
- ¿Cuánto oxígeno se consume? ¿Cuánto CO2 se produce?

### 4.1 Efecto de la fuente de carbono

In [ ]:
fuentes = {
    "Glucosa":  ("EX_glc__D_e", -10),
    "Glicerol": ("EX_glyc_e",   -10),
    "Piruvato": ("EX_pyr_e",    -10),
    "Acetato":  ("EX_ac_e",     -10),
    "Fumarato": ("EX_fum_e",    -10),
}

resultados = []
for nombre, (rxn_id, uptake) in fuentes.items():
    with model:
        model.reactions.get_by_id("EX_glc__D_e").lower_bound = 0
        try:
            model.reactions.get_by_id(rxn_id).lower_bound = uptake
            sol = model.optimize()
            mu = sol.objective_value if sol.status == 'optimal' else 0.0
        except Exception:
            mu = 0.0
        resultados.append((nombre, mu))

print(f"{'Fuente de carbono':<20} {'mu (1/h)':>10}  Barra")
print("-" * 55)
for nombre, mu in resultados:
    bar = chr(9608) * int(mu * 30)
    print(f"{nombre:<20} {mu:>10.4f}  {bar}")

### 4.2 Aerobio vs. anaerobio

**Preguntas antes de simular:**
- ¿Cómo cambia el metabolismo sin oxígeno?
- ¿Por qué se produce etanol en condiciones anaerobias pero no aerobias?
- ¿Es mejor crecer aerobio o anaerobio si quieres producir etanol?

In [ ]:
sol_aerobio = model.optimize()
print("Aerobio (O2 disponible):")
print(f"  Crecimiento:       {sol_aerobio.objective_value:.4f} 1/h")
print(f"  Consumo O2:        {sol_aerobio.fluxes.get('EX_o2_e', 0):.4f} mmol/gDW/h")
print(f"  Excreción acetato: {sol_aerobio.fluxes.get('EX_ac_e', 0):.4f} mmol/gDW/h")
print(f"  Excreción etanol:  {sol_aerobio.fluxes.get('EX_etoh_e', 0):.4f} mmol/gDW/h")

with model:
    model.reactions.get_by_id("EX_o2_e").lower_bound = 0
    sol_anaer = model.optimize()
    print("\nAnaerobio (sin O2):")
    print(f"  Crecimiento:       {sol_anaer.objective_value:.4f} 1/h")
    print(f"  Consumo O2:        {sol_anaer.fluxes.get('EX_o2_e', 0):.4f} mmol/gDW/h")
    print(f"  Excreción acetato: {sol_anaer.fluxes.get('EX_ac_e', 0):.4f} mmol/gDW/h")
    print(f"  Excreción etanol:  {sol_anaer.fluxes.get('EX_etoh_e', 0):.4f} mmol/gDW/h")
    print(f"  Excreción formiato:{sol_anaer.fluxes.get('EX_for_e', 0):.4f} mmol/gDW/h")
    print(f"  Excreción lactato: {sol_anaer.fluxes.get('EX_lac__D_e', 0):.4f} mmol/gDW/h")

## 📊 Parte 5: Dynamic Flux Balance Analysis (dFBA) en COBRApy

El **Análisis de Balance de Flujos Dinámico (dFBA)** es una extensión del FBA tradicional que permite simular el comportamiento de un sistema metabólico a lo largo del tiempo. A diferencia del FBA estático, que asume un estado estacionario, el dFBA integra la dinámica de los cambios en las concentraciones de metabolitos extracelulares y la biomasa.

Este método es útil para:
- Predecir la evolución de cultivos microbianos.
- Optimizar procesos de fermentación y producción de metabolitos.
- Entender cómo los cambios ambientales afectan el metabolismo a lo largo del tiempo.

El siguiente ejemplo muestra una implementación simple (aunque lenta) de dFBA utilizando COBRApy y `scipy.integrate.solve_ivp`. Este cuaderno presenta un enfoque de optimización estática (SOA) y no debe considerarse listo para producción.

El modelo considera únicamente un crecimiento básico limitado por Michaelis-Menten en glucosa.

El dFBA combina un sistema dinámico de concentraciones celulares externas con un modelo metabólico en estado pseudoestacionario.

En este notebook, definimos la función add_dynamic_bounds(model, y) para convertir las concentraciones externas de metabolitos en límites para los flujos de frontera del modelo metabólico. Estas funciones nos ayudarán con el dFBA. Entonces vamos a las modelaciones.

In [ ]:
model = load_model(model_to_use) # asegura el modelo sin modficiaciones

ts = np.linspace(0, 15, 100)  # Desired integration resolution and interval
y0 = [0.1, 10, 0, 0, 0, 0, 0] # Initial conditions for [biomass, glucose, ethanol, oxygen, succinate, lactate, acetate]

with tqdm() as pbar:
    dynamic_system.pbar = pbar

    sol = solve_ivp(
        fun=dynamic_system,
        events=[infeasible_event],
        t_span=(ts.min(), ts.max()),
        y0=y0,
        t_eval=ts,
        rtol=1e-6,
        atol=1e-8,
        method='BDF'
    )

sol

Dado que el cultivo se queda sin glucosa, la simulación finaliza antes de tiempo. El momento exacto de esta «muerte celular» se registra en sol.t_events.

### 5.1. Crear el grafico de tiempo vs biomasa y glucosa

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot Biomass on the left Y-axis
ax1.plot(sol.t, sol.y.T[:, 0], color='blue', label='Biomass')
ax1.set_ylabel('Biomass (gDW/L)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a twin Y-axis for Glucose, Ethanol, Oxygen, Succinate, Lactate, and Acetate
ax2 = ax1.twinx()
ax2.plot(sol.t, sol.y.T[:, 1], color='red', label='Glucose')
ax2.plot(sol.t, sol.y.T[:, 2], color='green', label='Ethanol')
ax2.plot(sol.t, np.abs(sol.y.T[:, 3]), color='purple', label='Oxygen') # Use absolute value for oxygen
ax2.plot(sol.t, sol.y.T[:, 4], color='brown', label='Succinate')
ax2.plot(sol.t, sol.y.T[:, 5], color='orange', label='Lactate')
ax2.plot(sol.t, sol.y.T[:, 6], color='gray', label='Acetate')
ax2.set_ylabel('Concentration (mmol/L)', color='black') # General label for the right axis
ax2.tick_params(axis='y', labelcolor='black')

# Set X-axis label and Title
ax1.set_xlabel('Time (h)')
ax1.set_title('Dynamic Simulation of Escherichia coli using E coli core model')
ax1.grid(True)

# Combine legends from both axes
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

plt.tight_layout()
plt.show()

Ahora veamos como es la producción bajo condiciones anaerobicas

In [ ]:
model = load_model(model_to_use) # asegura el modelo sin modficiaciones
model.reactions.get_by_id("EX_o2_e").lower_bound = 0

ts = np.linspace(0, 15, 100)  # Desired integration resolution and interval
y0 = [0.1, 10, 0, 0, 0, 0, 0] # Initial conditions for [biomass, glucose, ethanol, oxygen, succinate, lactate, acetate]

with tqdm() as pbar:
    dynamic_system.pbar = pbar

    sol = solve_ivp(
        fun=dynamic_system,
        events=[infeasible_event],
        t_span=(ts.min(), ts.max()),
        y0=y0,
        t_eval=ts,
        rtol=1e-6,
        atol=1e-8,
        method='BDF'
    )

sol

### 5.2. Crear el grafico de tiempo vs biomasa y glucosa en condiciones anaerobicas

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot Biomass on the left Y-axis
ax1.plot(sol.t, sol.y.T[:, 0], color='blue', label='Biomass')
ax1.set_ylabel('Biomass (gDW/L)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a twin Y-axis for Glucose, Ethanol, Oxygen, Succinate, Lactate, and Acetate
ax2 = ax1.twinx()
ax2.plot(sol.t, sol.y.T[:, 1], color='red', label='Glucose')
ax2.plot(sol.t, sol.y.T[:, 2], color='green', label='Ethanol')
ax2.plot(sol.t, np.abs(sol.y.T[:, 3]), color='purple', label='Oxygen') # Use absolute value for oxygen
ax2.plot(sol.t, sol.y.T[:, 4], color='brown', label='Succinate')
ax2.plot(sol.t, sol.y.T[:, 5], color='orange', label='Lactate')
ax2.plot(sol.t, sol.y.T[:, 6], color='gray', label='Acetate')
ax2.set_ylabel('Concentration (mmol/L)', color='black') # General label for the right axis
ax2.tick_params(axis='y', labelcolor='black')

# Set X-axis label and Title
ax1.set_xlabel('Time (h)')
ax1.set_title('Dynamic Simulation of Escherichia coli using E coli core model')
ax1.grid(True)

# Combine legends from both axes
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

plt.tight_layout()
plt.show()

## 🔎 Parte 6 - Diseño básico de biofábricas

### 6.1 Producción de etanol — cambiar la función objetivo

**Preguntas de anticipación:**
- ¿Qué reacciones crees que participan en la producción de etanol?
- ¿Será posible maximizar etanol Y crecimiento simultáneamente?
- ¿Cuál es el trade-off entre ambos objetivos?

In [ ]:
model = load_model(model_to_use)

print("Reacciones relacionadas con etanol:")
for rxn in model.reactions:
    if 'eth' in rxn.id.lower() or 'etoh' in rxn.id.lower():
        print(f"  {rxn.id}: {rxn.name} | {rxn.reaction}")

In [ ]:
with model:
    model.objective = "EX_etoh_e"
    sol_etanol = model.optimize()

    print("=" * 55)
    print("Maximizando producción de etanol")
    print("=" * 55)
    print(f"Estado:                       {sol_etanol.status}")
    print(f"Producción máxima de etanol:  {sol_etanol.objective_value:.4f} mmol/gDW/h")
    print(f"Tasa de crecimiento asociada: {sol_etanol.fluxes['Biomass_Ecoli_core']:.4f} 1/h")
    print("\nFlujos de vías centrales:")
    for rxn_id in ['PFK', 'PGI', 'PYK', 'PDH', 'CS', 'ACONTa']:
        try:
            flux = sol_etanol.fluxes[rxn_id]
            nombre = model.reactions.get_by_id(rxn_id).name
            print(f"  {rxn_id:<12} {nombre:<30} {flux:>10.4f}")
        except Exception:
            pass

### 6.2 Production Envelope — trade-off crecimiento vs. producción

In [ ]:
print("Calculando production envelope...")
envelope = production_envelope(
    model,
    reactions=["EX_etoh_e"],
    objective="Biomass_Ecoli_core",
    carbon_sources="EX_glc__D_e"
)
print(f"Columnas disponibles: {list(envelope.columns)}")
print(envelope.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# EX_etoh_e contiene los puntos del eje X (etanol barrido como variable independiente)
# flux_maximum/minimum son los valores de biomasa (objetivo)
etoh_axis = envelope['EX_etoh_e']        # eje X: producción de etanol
bio_min   = envelope['flux_minimum']     # eje Y mínimo: biomasa
bio_max   = envelope['flux_maximum']     # eje Y máximo: biomasa

ax.fill_between(etoh_axis, bio_min, bio_max,
                alpha=0.4, color='steelblue', label='Espacio fenotípico factible')
ax.plot(etoh_axis, bio_max, color='steelblue', linewidth=2, label='Máximo de crecimiento')

wt_sol = model.optimize()
ax.axhline(y=wt_sol.objective_value, color='green', linestyle='--', linewidth=1.5,
           label=f'mu_WT = {wt_sol.objective_value:.3f} 1/h')

ax.set_xlabel('Producción de etanol (mmol/gDW/h)', fontsize=12)
ax.set_ylabel('Tasa de crecimiento mu (1/h)', fontsize=12)
ax.set_title('Production Envelope: Etanol vs. Biomasa\n(E. coli core, glucosa aerobio)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

La figura anterior muestra como se da la producción de etanol en *E. coli* usando como fuente de carbono glucosa bajo condiciones aerobicas. Ahora, miremos como se comporta el uso de oxigeno.

In [ ]:
print("Calculando production envelope...")
envelope = production_envelope(
    model,
    reactions=["EX_o2_e"],
    objective="Biomass_Ecoli_core",
    carbon_sources="EX_glc__D_e"
)
print(f"Columnas disponibles: {list(envelope.columns)}")
print(envelope.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# EX_o2_e contiene los puntos del eje X (oxigeno barrido como variable independiente)
# flux_maximum/minimum son los valores de biomasa (objetivo)
o2_axis = -envelope['EX_o2_e']         # eje X: consumo de oxigeno, los valores de consumo son negativos asi que debemos pasarlos a positivos
bio_min = envelope['flux_minimum']     # eje Y mínimo: biomasa
bio_max = envelope['flux_maximum']     # eje Y máximo: biomasa

ax.fill_between(o2_axis, bio_min, bio_max,
                alpha=0.4, color='steelblue', label='Espacio fenotípico factible')
ax.plot(o2_axis, bio_max, color='steelblue', linewidth=2, label='Máximo de crecimiento')

wt_sol = model.optimize()
ax.axhline(y=wt_sol.objective_value, color='green', linestyle='--', linewidth=1.5,
           label=f'mu_WT = {wt_sol.objective_value:.3f} 1/h')

ax.set_xlabel('Consumo de oxigeno (mmol/gDW/h)', fontsize=12)
ax.set_ylabel('Tasa de crecimiento mu (1/h)', fontsize=12)
ax.set_title('Production Envelope: Oxigeno vs. Biomasa\n(E. coli core, glucosa)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

Finalmente, miremos el comportamiento en la producción de etanol, usando glucosa, pero esta vez bajo condiciones anaerobicas

In [ ]:
model.reactions.get_by_id("EX_o2_e").lower_bound = 0

print("Calculando production envelope...")
envelope = production_envelope(
    model,
    reactions=["EX_etoh_e"],
    objective="Biomass_Ecoli_core",
    carbon_sources="EX_glc__D_e"
)
print(f"Columnas disponibles: {list(envelope.columns)}")
print(envelope.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# EX_etoh_e contiene los puntos del eje X (etanol barrido como variable independiente)
# flux_maximum/minimum son los valores de biomasa (objetivo)
etoh_axis = envelope['EX_etoh_e']        # eje X: producción de etanol
bio_min   = envelope['flux_minimum']     # eje Y mínimo: biomasa
bio_max   = envelope['flux_maximum']     # eje Y máximo: biomasa

ax.fill_between(etoh_axis, bio_min, bio_max,
                alpha=0.4, color='steelblue', label='Espacio fenotípico factible')
ax.plot(etoh_axis, bio_max, color='steelblue', linewidth=2, label='Máximo de crecimiento')

wt_sol = model.optimize()
ax.axhline(y=wt_sol.objective_value, color='green', linestyle='--', linewidth=1.5,
           label=f'mu_WT = {wt_sol.objective_value:.3f} 1/h')

ax.set_xlabel('Producción de etanol (mmol/gDW/h)', fontsize=12)
ax.set_ylabel('Tasa de crecimiento mu (1/h)', fontsize=12)
ax.set_title('Production Envelope: Etanol vs. Biomasa\n(E. coli core, glucosa aerobio)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 🧫 Parte 7 - Extensión — Circuito sintético: Producción de insulina en *E. coli*

Un aspecto adicional que se puede modelar, es la carga al modificar geneticamente un microorganismo. Especialmente cuando se agregan nuevos genes mediante circuitos.

En este sentido, una bacteria *E. coli* a la que le agregamos el gen de la insulina humana crece más lento que la original. ¿Por qué? ¿Cuánto más lento? ¿Se podría predecir sin hacer el experimento?

Al final de este cuaderno vamos a contestar esa pregunta con las mismas herramientas que vamos a construir: **balances de masa ($\mathbf{S}\cdot\mathbf{v}=0$), ODEs y FBA**.

Ahora simulamos un escenario de biología sintética. Un plasmido que:

1. Contiene el gen de insulina humana bajo control del operón lac
2. Se induce con IPTG (isopropil β-D-1-tiogalactopiranósido)
3. Produce insulina recombinante


**El circuito (contexto biológico)**
- Plásmido tipo pUC (de **alto** número de copias) con el gen de la insulina humana bajo el promotor P_tac, inducible con IPTG.
- Preproinsulina humana: 110 aminoácidos (~12 kDa). La insulina madura tiene 51 aminoácidos (cadena A de 21 y cadena B de 30, ~5.8 kDa).
- En *E. coli* normalmente se produce como proinsulina o proteína de fusión (a menudo en cuerpos de inclusión) y se procesa *in vitro*.

**Idea del modelo:** sintetizar una proteína heteróloga cuesta **ATP y aminoácidos**, los mismos recursos que usa la biomasa. Añadimos una reacción que representa ese costo, y el FBA nos dice cuánto crecimiento se sacrifica.

**Preguntas de anticipación**
1. ¿Qué pasará con el crecimiento cuando expresamos insulina?
2. ¿Cuál es el "costo metabólico" de producir una proteína heteróloga?
3. ¿Cuánto de la capacidad de síntesis de proteína puede tomar la célula antes de dejar de crecer?
4. ¿Es mejor la expresión constitutiva o la inducible?

In [ ]:
ec = load_model("textbook")            # E. coli core (modelo del libro)
OBJ = "Biomass_Ecoli_core"
ec.objective = OBJ

# Reacción aproximada de síntesis de proteína heteróloga ("equivalente de aminoácido")
# El modelo core solo tiene Glu y Gln, así que se usan como aminoácido promedio.
insulina = Reaction("insulina", name="proteína heteróloga (aprox.)", lower_bound=0, upper_bound=1000)
ec.add_reactions([insulina])
insulina.add_metabolites({
    ec.metabolites.glu__L_c: -0.5, ec.metabolites.gln__L_c: -0.5,    # aminoácido promedio
    ec.metabolites.atp_c: -4, ec.metabolites.h2o_c: -4,              # ~4 ATP por enlace peptídico
    ec.metabolites.adp_c: 4, ec.metabolites.pi_c: 4, ec.metabolites.h_c: 4,
})
print("Reacción añadida:", insulina.reaction)

# Síntesis proteica total del WT (mmol aminoácido / gDW / h)
# Supuestos: 55 % del peso seco es proteína; ~110 g/mol por aminoácido
mu_wt = ec.slim_optimize()
P_SYN = 0.55 * mu_wt / 0.110
print(f"Crecimiento WT: {mu_wt:.3f} 1/h | síntesis proteica total ≈ {P_SYN:.2f} mmol aa/gDW/h")

# Escenarios: fracción de la síntesis proteica dedicada a la proteína heteróloga
escenarios = {
    "Sin plásmido (WT)":    0.00,
    "Plásmido sin inducir": 0.01,   # expresión basal ("fuga")
    "Plásmido + IPTG":      0.20,   # inducido
}

filas = []
for nombre, fraccion in escenarios.items():
    flujo = fraccion * P_SYN
    with ec:
        ec.reactions.insulina.bounds = (flujo, flujo)
        sol = ec.optimize()
    filas.append({
        "Escenario": nombre,
        "% síntesis proteica a insulina": 100 * fraccion,
        "Flujo insulina (mmol aa/gDW/h)": round(flujo, 3),
        "ATP gastado en insulina (mmol/gDW/h)": round(4 * flujo, 2),
        "Crecimiento (1/h)": round(sol.objective_value, 4),
        "Caída de crecimiento (%)": round(100 * (1 - sol.objective_value / mu_wt), 1),
    })

resultados = pd.DataFrame(filas)
display(resultados)


📖 **Análisis: ¿qué hicimos y qué muestra la tabla?**


**Qué hicimos.** Al modelo de *E. coli* le agregamos **una reacción** que representa fabricar la proteína heteróloga. Esa reacción gasta lo mismo que gasta la biomasa: **aminoácidos y ATP** (~4 ATP por enlace peptídico). Luego fijamos cuánto flujo pasa por ella en cada escenario y dejamos que el FBA maximice el crecimiento con lo que queda.

**Cómo definimos los escenarios.** No fijamos un flujo cualquiera: dijimos "la célula dedica el X % de su síntesis de proteína a la insulina". El 100 % de referencia es la síntesis proteica total del WT (≈ 4.4 mmol aminoácido/gDW/h), que calculamos suponiendo que el 55 % del peso seco es proteína.

**Cómo leer la tabla**

| Escenario | Insulina | ATP gastado en insulina | Crecimiento | Caída |
|:--|:--:|:--:|:--:|:--:|
| WT | 0 % | 0 | 0.874 | 0 % |
| Sin inducir | 1 % | ≈ 0.17 | 0.870 | ≈ 0.5 % |
| Con IPTG | 20 % | ≈ 3.5 | 0.793 | ≈ 9.3 % |

- La **caída de crecimiento** es proporcional a la cantidad de insulina: pasar de 1 % a 20 % multiplica el costo por 20.
- Fíjate en el ATP: con IPTG, la insulina gasta ≈ 3.5 mmol ATP/gDW/h, es decir, cerca del **40 % del ATP que la célula ya gasta solo en mantenimiento** (8.39). Es un gasto que antes se usaba para crecer.
- La glucosa disponible es la misma en todos los escenarios (10 mmol/gDW/h). Lo que cambia es **en qué se gasta**.

**Preguntas**
- ¿Por qué la expresión basal (1 %) casi no afecta el crecimiento, y por qué con IPTG (20 %) sí se nota?
- ¿Qué dos recursos compiten entre la biomasa y la insulina en este modelo?
- En la tabla, el ATP gastado en insulina es de ≈ 3.5 con IPTG. ¿Por qué eso no reduce el crecimiento en un 40 %, sino solo en ≈ 9 %?
- Si la glucosa disponible se redujera a la mitad, ¿esperarías que la caída porcentual de crecimiento fuera mayor, menor o igual? Justifica y comprueba cambiando `EX_glc__D_e`.
- ¿Qué supuestos hicimos para definir el "100 %" de síntesis proteica? ¿Cómo cambiaría la tabla si fueran otros?

### 7.1 Visualizar el trade-off

In [ ]:
# Envolvente de producción: máxima proteína heteróloga para cada nivel de crecimiento
ec.objective = "insulina"
max_ins = ec.slim_optimize()
ec.objective = OBJ

flujos = np.linspace(0, max_ins, 30)
crec = []
for f in flujos:
    with ec:
        ec.reactions.insulina.bounds = (f, f)
        crec.append(ec.slim_optimize())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: crecimiento por escenario
nombres = resultados["Escenario"]
vals = resultados["Crecimiento (1/h)"]
ax1.bar(range(len(nombres)), vals, color=["green", "orange", "red"], alpha=0.7, edgecolor="black")
for i, (v_, c_) in enumerate(zip(vals, resultados["Caída de crecimiento (%)"])):
    ax1.text(i, v_ + 0.01, f"{v_:.3f}\n(−{c_}%)", ha="center", va="bottom", fontsize=10)
ax1.set_xticks(range(len(nombres))); ax1.set_xticklabels(nombres, rotation=15, ha="right")
ax1.set_ylabel("Tasa de crecimiento (1/h)"); ax1.set_ylim(0, 1.05)
ax1.set_title("Crecimiento según la expresión de insulina"); ax1.grid(axis="y", alpha=.3)

# Panel 2: envolvente de producción
ax2.plot(crec, flujos, "-", color="tab:blue", lw=2)
ax2.fill_between(crec, flujos, color="tab:blue", alpha=.12)
for nombre, color in zip(nombres, ["green", "orange", "red"]):
    fila = resultados[resultados["Escenario"] == nombre].iloc[0]
    ax2.plot(fila["Crecimiento (1/h)"], fila["Flujo insulina (mmol aa/gDW/h)"], "o", color=color, ms=10, label=nombre)
ax2.set_xlabel("Crecimiento (1/h)"); ax2.set_ylabel("Flujo de proteína heteróloga (mmol aa/gDW/h)")
ax2.set_title("Envolvente de producción: crecer o producir"); ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()
print(f"Máxima producción posible (sin crecer): {max_ins:.2f} mmol aa/gDW/h")


**Análisis: lectura de las gráficas**

**Panel izquierdo (barras).** Es la tabla de la Celda 1 dibujada. Las tres barras bajan de forma escalonada: sin plásmido (verde), con plásmido sin inducir (naranja, casi igual al WT) y con IPTG (rojo, ≈ 9 % menos).

**Panel derecho (envolvente de producción).** Muestra **todas** las combinaciones posibles de crecimiento y producción, no solo tres escenarios. Cada punto de la recta es un reparto distinto de los mismos recursos:

- **Extremo derecho:** la célula crece al máximo (0.874 1/h) y no fabrica nada de proteína heteróloga.
- **Extremo izquierdo:** la célula dedica todo a la proteína heteróloga (≈ 9.4 mmol aa/gDW/h) y **no crece**.
- **Entre medias:** por cada unidad de insulina que se agrega, el crecimiento baja siempre la misma cantidad, por eso es una recta. Es el mismo compromiso que vimos con el mini-modelo (donde biomasa = 10 − A secretada): dos salidas que se reparten un recurso limitado.
- **Los tres puntos de colores** son los escenarios de la tabla, ubicados sobre esa recta. Todos están en la parte alta, a la derecha: el costo de la insulina es pequeño frente a lo que la célula podría producir en total.
- **El área sombreada** son las combinaciones alcanzables; todo lo que queda arriba de la recta es inalcanzable con esos recursos.

**Preguntas**

- ¿Por qué el trade-off es una **recta** y no una curva? (Pista: ¿cuántos recursos limitan el sistema y cómo entran en las ecuaciones?)
- En el extremo izquierdo, la célula produce ≈ 9.4 mmol aa/gDW/h, más del doble de la síntesis proteica total del WT (≈ 4.4). ¿Eso es realista? ¿Qué le falta al modelo para decirlo?
- Si en lugar de 20 % se dedicara 50 % de la síntesis proteica a la insulina, ¿en qué zona de la gráfica caería el punto y cuánto crecimiento quedaría?
- En un cultivo en dos fases (crecer, luego inducir), ¿qué punto de la envolvente buscas en cada fase y por qué?
- Experimentalmente, expresar una proteína al 20 % suele frenar el crecimiento mucho más de un 9 %. ¿Qué crees que el modelo **no está viendo**?
- Si dispones de una cepa que produce 20 % más ATP por glucosa, ¿cómo cambiaría la recta? ¿Se movería su extremo izquierdo, el derecho o ambos?

In [ ]:
flujo_fijo = 0.20 * P_SYN      # mismo flujo absoluto de proteína heteróloga
filas = []
for cond, lb_o2 in [("Aerobio", -1000), ("Anaerobio", 0)]:
    with ec:
        ec.reactions.EX_o2_e.lower_bound = lb_o2
        mu0 = ec.slim_optimize()
        ec.reactions.insulina.bounds = (flujo_fijo, flujo_fijo)
        mu1 = ec.slim_optimize()
    filas.append({"Condición": cond, "Crecimiento WT": round(mu0, 3),
                  "Con insulina (20 %)": round(mu1, 3),
                  "Caída (%)": round(100 * (1 - mu1 / mu0), 1)})
display(pd.DataFrame(filas))

## 🎯 Parte 8 - Simulación de knockouts de genes

### 8.1 Knockout simple — gen pgi (fosfoglucosa isomerasa)

Este gen conecta glucólisis con la vía de pentosa fosfato.
Al bloquearlo, el carbono de G6P se redirige hacia esa vía.

**Preguntas antes de simular:**
- ¿Crees que pgi es un gen esencial?
- ¿Qué rutas alternativas podría usar la célula si pgi se bloquea?
- ¿Cuánto se reduciría el crecimiento?

Una pregunta habitual al analizar modelos metabólicos es qué pasaría si se impidiera por completo el flujo de una reacción determinada. Esto se puede comprobar utilizando cobrapy mediante

In [ ]:
model.reactions.get_by_id("PGI")

In [ ]:
model = load_model(model_to_use) # asegura el modelo sin modficiaciones

wt_solution = model.optimize()
print("Cepa silvestre (WT):")
print(f"  Crecimiento:   {wt_solution.objective_value:.4f} 1/h")
print(f"  Flujo PGI:     {wt_solution.fluxes['PGI']:.4f} mmol/gDW/h")
print(f"  Flujo G6PDH2r: {wt_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.reactions.get_by_id("PGI").knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-PFK (knockout reaction):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo PGI:     {ko_solution.fluxes['PGI']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {ko_solution.fluxes.get('G6PDH2r', 0):.4f} mmol/gDW/h")

reduccion = (1 - ko_solution.objective_value / wt_solution.objective_value) * 100
print(f"\nReducción en crecimiento: {reduccion:.1f}%")

Para evaluar estrategias de manipulación genética, resulta más interesante analizar qué ocurre si se inactivan determinados genes, ya que ello puede provocar la ausencia de reacciones en caso de redundancia, o bien un aumento de las reacciones si el gen en cuestión participa en más de una reacción.

In [ ]:
model = load_model(model_to_use) # asegura el modelo sin modficiaciones

wt_solution = model.optimize()
print("Cepa silvestre (WT):")
print(f"  Crecimiento:   {wt_solution.objective_value:.4f} 1/h")
print(f"  Flujo PGI:     {wt_solution.fluxes['PGI']:.4f} mmol/gDW/h")
print(f"  Flujo G6PDH2r: {wt_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.genes.b4025.knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-pgi (knockout gene):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo pgi:     {ko_solution.fluxes['PGI']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {ko_solution.fluxes.get('G6PDH2r', 0):.4f} mmol/gDW/h")

En este caso PGI tiene un solo gen y participa solo en esta reacción. Pero miremos al fosfofructo kinasa - PFK, es el sigueinte paso en la glucólisis despues de PGI

In [ ]:
model.reactions.get_by_id("PFK")

In [ ]:
model = load_model(model_to_use) # asegura el modelo sin modficiaciones

wt_solution = model.optimize()
print("Cepa silvestre (WT):")
print(f"  Crecimiento:   {wt_solution.objective_value:.4f} 1/h")
print(f"  Flujo PFK:     {wt_solution.fluxes['PFK']:.4f} mmol/gDW/h")
print(f"  Flujo G6PDH2r: {wt_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.reactions.get_by_id("PFK").knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-PFK (knockout reacción):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo PFK:     {ko_solution.fluxes['PFK']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {ko_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.genes.b1723.knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-pfkA (knockout gene):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo pfkA:    {ko_solution.fluxes['PFK']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {ko_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.genes.b3916.knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-pfkB (knockout gene):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo pfkB:    {ko_solution.fluxes['PFK']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {ko_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

with model:
    model.genes.b1723.knock_out()
    model.genes.b3916.knock_out()
    ko_solution = model.optimize()
    print("\nCepa delta-pfkAB (knockout gene):")
    print(f"  Crecimiento:   {ko_solution.objective_value:.4f} 1/h")
    print(f"  Flujo pfkAB:   {ko_solution.fluxes['PFK']:.4f} mmol/gDW/h")
    print(f"  Flujo G6PDH2r: {wt_solution.fluxes.get('G6PDH2r'):.4f} mmol/gDW/h")

### 8.2 Esencialidad génica — knockouts sistemáticos

In [ ]:
from cobra.flux_analysis import single_reaction_deletion

print("Ejecutando knockouts sistemáticos de reacciones...")
deletion_results = single_reaction_deletion(model)

wt_growth = model.optimize().objective_value
threshold = 0.01

esenciales = deletion_results[deletion_results['growth'] < threshold]
reducidas  = deletion_results[(deletion_results['growth'] >= threshold) &
                                (deletion_results['growth'] < wt_growth * 0.99)]
sin_efecto = deletion_results[deletion_results['growth'] >= wt_growth * 0.99]

print(f"\nTotal reacciones evaluadas: {len(deletion_results)}")
print(f"  Esenciales (letales):    {len(esenciales):>4}  ({len(esenciales)/len(deletion_results)*100:.1f}%)")
print(f"  Reducen crecimiento:     {len(reducidas):>4}  ({len(reducidas)/len(deletion_results)*100:.1f}%)")
print(f"  Sin efecto:              {len(sin_efecto):>4}  ({len(sin_efecto)/len(deletion_results)*100:.1f}%)")

In [ ]:
# Mostrar reacciones esenciales con sus genes asociados
print("Reacciones esenciales y sus genes")
print("\n")
for rxn_set in esenciales['ids']:
    rxn_id = list(rxn_set)[0]
    rxn = model.reactions.get_by_id(rxn_id)
    genes = [g.id for g in rxn.genes]
    genes_str = ", ".join(genes) if genes else "Sin genes asociados"
    print(f"  {rxn_id:<30} | Genes: {genes_str}")

print("\n")

# Mostrar reacciones que reducen crecimiento con sus genes
print("Reacciones que reducen el crecimiento y sus genes")
print("\n")
for rxn_set in reducidas['ids']:
    rxn_id = list(rxn_set)[0]
    rxn = model.reactions.get_by_id(rxn_id)
    genes = [g.id for g in rxn.genes]
    genes_str = ", ".join(genes) if genes else "Sin genes asociados"
    # Need to get the growth value from the original 'reducidas' DataFrame
    # The index of 'reducidas' DataFrame is still integer, so we find the row by its 'ids' column
    growth_val = reducidas[reducidas['ids'] == rxn_set]['growth'].iloc[0]
    print(f"  {rxn_id:<30} | Growth: {growth_val:.4f} | Genes: {genes_str}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
growth_values = deletion_results['growth'].fillna(0)
ax.hist(growth_values, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=wt_growth, color='green', linestyle='--', linewidth=2,
           label=f'WT: {wt_growth:.3f} 1/h')
ax.axvline(x=threshold, color='red', linestyle='--', linewidth=2,
           label='Umbral esencialidad')
ax.set_xlabel('Tasa de crecimiento tras knockout (1/h)', fontsize=12)
ax.set_ylabel('Número de reacciones', fontsize=12)
ax.set_title('Distribución de tasas de crecimiento\nen knockouts simples', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 🧪 Parte 9: Retos

Complete los siguientes ejercicios:

### Reto 1:

Calcule la tasa máxima de producción de succinato (EX_succ_e) en condiciones aerobias y anaerobias.

- ¿En cuál condición se produce más succinato?

In [ ]:
# Su código aquí:

### Reto 2

¿Cuántos genes son esenciales para el crecimiento en glucosa bajo condiciones aerobias? Use single_gene_deletion().

Liste los 5 primeros genes esenciales encontrados.


In [ ]:
# Su código aquí:

### Reto 3

Construya el production envelope para SUCCINATO vs. BIOMASA en condiciones ANAEROBIAS.

- ¿Qué estrategia es más conveniente para cada producto?

In [ ]:
# Su código aquí:

## Parte 10: Conexión con GECKO

### 10.1 El efecto de las restricciones enzimáticas

Hasta ahora trabajamos con FBA clásico. GECKO agrega restricciones que limitan cada flujo según
la cantidad de enzima disponible y su kcat. El siguiente código ilustra conceptualmente
cómo una restricción enzimática afecta la predicción:

In [ ]:
# Restricción enzimática para PDH (piruvato deshidrogenasa)
kcat_PDH   = 50           # 1/s
kcat_PDH_h = kcat_PDH * 3600   # convertir a 1/h
E_PDH      = 0.0001       # mmol enzima / gDW
v_max_PDH  = kcat_PDH_h * E_PDH

print("Restricción enzimática para PDH:")
print(f"  kcat    = {kcat_PDH} 1/s  =  {kcat_PDH_h:.0f} 1/h")
print(f"  [E_PDH] = {E_PDH} mmol/gDW")
print(f"  v_PDH <= kcat x [E] = {v_max_PDH:.1f} mmol/gDW/h")

sol_fba    = model.optimize()
v_PDH_fba  = sol_fba.fluxes.get('PDH', 0)
print(f"\nFBA clásico predice: v_PDH = {v_PDH_fba:.2f} mmol/gDW/h")

if abs(v_PDH_fba) > v_max_PDH:
    print(f"  -> Este flujo EXCEDE el límite enzimático ({v_max_PDH:.1f})")
    print(f"     GECKO corregiría esto imponiendo ub_PDH = {v_max_PDH:.1f}")
else:
    print(f"  -> El flujo está dentro del límite enzimático ({v_max_PDH:.1f})")

with model:
    model.reactions.get_by_id('PDH').upper_bound = v_max_PDH
    sol_gecko = model.optimize()
    diff = sol_fba.objective_value - sol_gecko.objective_value
    print(f"\nFBA con restricción enzimática en PDH:")
    print(f"  Crecimiento: {sol_gecko.objective_value:.4f} 1/h  (WT: {sol_fba.objective_value:.4f})")
    print(f"  Diferencia:  -{diff:.4f} 1/h")

### 10.2 Recursos para trabajar con GECKO

| Recurso | URL |
|:---|:---|
| GECKO GitHub | https://github.com/SysBioChalmers/GECKO |
| Modelos ecGEM listos | https://github.com/SysBioChalmers/ecModels |
| Tutorial GECKO | https://gecko.readthedocs.io |
| Artículo GECKO original | Sánchez et al., 2017, *Mol Syst Biol* |
| GECKO v3 | Domenzain et al., 2022, *Nat Commun* |

> **Recuerde:** Los modelos predicen; los experimentos validan.

## Conexión con el módulo

```
Genoma anotado          ->  model.genes  (reglas GPR)
Reacciones bioquímicas  ->  model.reactions + model.metabolites
Matriz estequiométrica  ->  create_stoichiometric_matrix(model)  -> S·v = 0
FBA                     ->  model.optimize()
Knockout                ->  rxn.knock_out() / single_gene_deletion()
Diseño de biofábrica    ->  cambiar objetivo + production_envelope()
Restricción enzimática  ->  v_i <= kcat_i · [E_i]  (GECKO)